# Silver Macro Transform

Macro-domain Silver notebook with two source-specific branches:

- `source_system=ecb`: thin notebook path delegated to `src/lakehouse`
- `source_system=fred`: heavy notebook path for phase-1 validation

The FRED branch performs technical duplicate elimination on the full revision key `(series_id, observation_date, realtime_start, realtime_end)` and does not collapse revisions into a latest-only view.

Rejected FRED rows are appended to `slv_macro.fred_series_clean_quarantine` using standardized `dq_reason` codes.

In [ ]:
import sys
import uuid
from datetime import datetime
from pathlib import Path

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.common.runtime import UTC, parse_series_ids, resolve_date_window
from lakehouse.pipelines.silver import run_silver_ecb_fx_ref_rates_daily


def collect_counts(df, key_column: str, count_alias: str) -> dict[str, int]:
    counts = (
        df.groupBy(key_column)
        .count()
        .withColumnRenamed("count", count_alias)
        .collect()
    )
    return {row[key_column]: int(row[count_alias]) for row in counts}


spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("source_system", "ecb")
dbutils.widgets.text("quote_currencies", "USD")
dbutils.widgets.text("series_ids", "CPIAUCSL,FEDFUNDS,GDP")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
source_system = dbutils.widgets.get("source_system").strip().lower()
raw_lookback_days = dbutils.widgets.get("lookback_days").strip()
lookback_days_raw = raw_lookback_days or ("90" if source_system == "fred" else "7")
mode = dbutils.widgets.get("mode").strip().lower()
run_id = dbutils.widgets.get("run_id").strip()

if source_system == "ecb":
    result = run_silver_ecb_fx_ref_rates_daily(
        spark=spark,
        raw_quote_currencies=dbutils.widgets.get("quote_currencies").strip(),
        mode=mode,
        start_date_raw=dbutils.widgets.get("start_date").strip(),
        end_date_raw=dbutils.widgets.get("end_date").strip(),
        lookback_days_raw=lookback_days_raw,
        run_id=run_id,
        catalog=catalog,
        display_fn=display,
    ).as_dict()
elif source_system == "fred":
    source_table = f"{catalog}.brz_macro.raw_fred_series"
    metadata_table = f"{catalog}.brz_macro.raw_fred_series_metadata"
    target_table = f"{catalog}.slv_macro.fred_series_clean"
    quarantine_table = f"{catalog}.slv_macro.fred_series_clean_quarantine"
    dedup_strategy = "technical_duplicate_elimination_on_full_revision_key"
    structural_reasons = [
        "MISSING_SERIES_ID",
        "MISSING_OBSERVATION_DATE",
        "MISSING_REALTIME_START",
        "MISSING_REALTIME_END",
        "INVALID_REALTIME_ORDER",
    ]

    for table_name in [source_table, metadata_table, target_table, quarantine_table]:
        if not spark.catalog.tableExists(table_name):
            raise RuntimeError(
                f"Required table {table_name} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
            )

    series_ids = parse_series_ids(dbutils.widgets.get("series_ids").strip())
    start_date, end_date = resolve_date_window(
        mode=mode,
        start_date_raw=dbutils.widgets.get("start_date").strip(),
        end_date_raw=dbutils.widgets.get("end_date").strip(),
        lookback_days_raw=lookback_days_raw,
    )
    silver_ingested_at = datetime.now(UTC)

    bronze_df = (
        spark.table(source_table)
        .select(
            "series_id",
            "observation_date",
            "realtime_start",
            "realtime_end",
            "value_raw",
            "value",
            "ingested_at",
            "run_id",
            "payload_hash",
        )
        .filter(F.col("series_id").isin(series_ids))
        .filter(
            (F.col("observation_date") >= F.lit(start_date))
            & (F.col("observation_date") <= F.lit(end_date))
        )
    )

    available_metadata_ids = {
        row["series_id"]
        for row in (
            spark.table(metadata_table)
            .select("series_id")
            .filter(F.col("series_id").isin(series_ids))
            .dropDuplicates()
            .collect()
        )
    }
    missing_metadata_ids = sorted(set(series_ids) - available_metadata_ids)
    metadata_complete = len(missing_metadata_ids) == 0

    rows_read = bronze_df.count()
    per_series_rows_read = collect_counts(bronze_df, "series_id", "rows_read") if rows_read else {}

    if rows_read == 0:
        result = {
            "status": "success_empty",
            "source_system": source_system,
            "mode": mode,
            "dedup_strategy": dedup_strategy,
            "series_ids": series_ids,
            "series_ids_missing_metadata": missing_metadata_ids,
            "metadata_complete": metadata_complete,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_table": source_table,
            "metadata_table": metadata_table,
            "target_table": target_table,
            "quarantine_table": quarantine_table,
            "rows_read": 0,
            "rows_after_dedup": 0,
            "rows_structural_invalid": 0,
            "rows_rejected": 0,
            "rows_quarantined": 0,
            "rows_to_update": 0,
            "rows_to_insert": 0,
            "rows_merged": 0,
            "run_id": run_id,
            "per_series_rows_read": per_series_rows_read,
            "per_series_rows_after_dedup": {},
            "per_series_rows_rejected": {},
            "per_series_rows_merged": {},
        }
    else:
        dedup_window = Window.partitionBy(
            "series_id",
            "observation_date",
            "realtime_start",
            "realtime_end",
        ).orderBy(
            F.col("ingested_at").desc(),
            F.col("payload_hash").desc(),
        )

        bronze_latest_df = (
            bronze_df
            .withColumn("_row_number", F.row_number().over(dedup_window))
            .filter(F.col("_row_number") == 1)
            .drop("_row_number")
        )

        rows_after_dedup = bronze_latest_df.count()
        per_series_rows_after_dedup = collect_counts(
            bronze_latest_df,
            "series_id",
            "rows_after_dedup",
        )

        if available_metadata_ids:
            metadata_flags_df = (
                spark.createDataFrame(
                    [(series_id,) for series_id in sorted(available_metadata_ids)],
                    "series_id string",
                )
                .withColumn("metadata_present", F.lit(True))
            )
            assessed_df = bronze_latest_df.join(metadata_flags_df, on="series_id", how="left")
        else:
            assessed_df = bronze_latest_df.withColumn("metadata_present", F.lit(None).cast("boolean"))

        dq_reason = (
            F.when(F.col("series_id").isNull(), F.lit("MISSING_SERIES_ID"))
            .when(F.col("observation_date").isNull(), F.lit("MISSING_OBSERVATION_DATE"))
            .when(F.col("realtime_start").isNull(), F.lit("MISSING_REALTIME_START"))
            .when(F.col("realtime_end").isNull(), F.lit("MISSING_REALTIME_END"))
            .when(F.col("realtime_start") > F.col("realtime_end"), F.lit("INVALID_REALTIME_ORDER"))
            .when(F.col("metadata_present").isNull(), F.lit("MISSING_METADATA"))
            .when(F.col("value").isNull(), F.lit("NULL_VALUE"))
        )

        assessed_df = assessed_df.withColumn("dq_reason", dq_reason)
        rows_structural_invalid = assessed_df.filter(F.col("dq_reason").isin(structural_reasons)).count()
        rejected_df = assessed_df.filter(F.col("dq_reason").isNotNull())
        rows_rejected = rejected_df.count()
        per_series_rows_rejected = (
            collect_counts(rejected_df, "series_id", "rows_rejected") if rows_rejected else {}
        )

        quarantine_df = rejected_df.select(
            F.lit(run_id).alias("run_id"),
            "series_id",
            "observation_date",
            "realtime_start",
            "realtime_end",
            "value_raw",
            "value",
            "dq_reason",
            F.col("ingested_at").alias("source_ingested_at"),
            F.col("run_id").alias("source_run_id"),
            "payload_hash",
            F.lit(silver_ingested_at).alias("quarantined_at"),
        )

        rows_quarantined = quarantine_df.count()
        if rows_quarantined > 0:
            quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

        valid_df = (
            assessed_df
            .filter(F.col("dq_reason").isNull())
            .select(
                "series_id",
                "observation_date",
                "realtime_start",
                "realtime_end",
                "value",
            )
            .withColumn("ingested_at", F.lit(silver_ingested_at))
            .withColumn("run_id", F.lit(run_id))
        )

        rows_valid = valid_df.count()
        per_series_rows_merged = (
            collect_counts(valid_df, "series_id", "rows_merged") if rows_valid else {}
        )

        if rows_valid == 0:
            if rows_quarantined > 0:
                display(quarantine_df.orderBy("series_id", "observation_date", "realtime_start", "dq_reason"))

            result = {
                "status": "success_empty_valid",
                "source_system": source_system,
                "mode": mode,
                "dedup_strategy": dedup_strategy,
                "series_ids": series_ids,
                "series_ids_missing_metadata": missing_metadata_ids,
                "metadata_complete": metadata_complete,
                "start_date": start_date.isoformat(),
                "end_date": end_date.isoformat(),
                "source_table": source_table,
                "metadata_table": metadata_table,
                "target_table": target_table,
                "quarantine_table": quarantine_table,
                "rows_read": rows_read,
                "rows_after_dedup": rows_after_dedup,
                "rows_structural_invalid": rows_structural_invalid,
                "rows_rejected": rows_rejected,
                "rows_quarantined": rows_quarantined,
                "rows_to_update": 0,
                "rows_to_insert": 0,
                "rows_merged": 0,
                "run_id": run_id,
                "per_series_rows_read": per_series_rows_read,
                "per_series_rows_after_dedup": per_series_rows_after_dedup,
                "per_series_rows_rejected": per_series_rows_rejected,
                "per_series_rows_merged": per_series_rows_merged,
            }
        else:
            existing_key_count = (
                valid_df.select("series_id", "observation_date", "realtime_start", "realtime_end")
                .join(
                    spark.table(target_table).select(
                        "series_id",
                        "observation_date",
                        "realtime_start",
                        "realtime_end",
                    ),
                    on=["series_id", "observation_date", "realtime_start", "realtime_end"],
                    how="inner",
                )
                .count()
            )

            DeltaTable.forName(spark, target_table).alias("tgt").merge(
                valid_df.alias("src"),
                "tgt.series_id = src.series_id AND "
                "tgt.observation_date = src.observation_date AND "
                "tgt.realtime_start = src.realtime_start AND "
                "tgt.realtime_end = src.realtime_end",
            ).whenMatchedUpdate(
                set={
                    "value": "src.value",
                    "ingested_at": "src.ingested_at",
                    "run_id": "src.run_id",
                }
            ).whenNotMatchedInsertAll().execute()

            display(valid_df.orderBy("series_id", "observation_date", "realtime_start", "realtime_end"))
            if rows_quarantined > 0:
                display(quarantine_df.orderBy("series_id", "observation_date", "realtime_start", "dq_reason"))

            result = {
                "status": "success",
                "source_system": source_system,
                "mode": mode,
                "dedup_strategy": dedup_strategy,
                "series_ids": series_ids,
                "series_ids_missing_metadata": missing_metadata_ids,
                "metadata_complete": metadata_complete,
                "start_date": start_date.isoformat(),
                "end_date": end_date.isoformat(),
                "source_table": source_table,
                "metadata_table": metadata_table,
                "target_table": target_table,
                "quarantine_table": quarantine_table,
                "rows_read": rows_read,
                "rows_after_dedup": rows_after_dedup,
                "rows_structural_invalid": rows_structural_invalid,
                "rows_rejected": rows_rejected,
                "rows_quarantined": rows_quarantined,
                "rows_to_update": existing_key_count,
                "rows_to_insert": rows_valid - existing_key_count,
                "rows_merged": rows_valid,
                "run_id": run_id,
                "per_series_rows_read": per_series_rows_read,
                "per_series_rows_after_dedup": per_series_rows_after_dedup,
                "per_series_rows_rejected": per_series_rows_rejected,
                "per_series_rows_merged": per_series_rows_merged,
            }
else:
    raise ValueError("source_system must be one of: ecb, fred")

result
